In [1]:
%pip install google-generativeai datasets pandas tqdm scikit-learn python-dotenv

import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold
from datasets import load_dataset
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import time
import os
import json
import uuid
import re
from tqdm import tqdm
from dotenv import load_dotenv

# Load API key from your .env file
load_dotenv()
API_KEY = os.getenv("API_KEY")

Note: you may need to restart the kernel to use updated packages.


/Users/donovanchen/UCI/CS175/RedditSentimentAnalysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2p/xnjv139d2ng8q9pmjm0k0w9m0000gn/T/ipykernel_43436/1032341030.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# 1. Load the dataset
goEmotionsDataset = load_dataset('go_emotions')
sentimentLabels = goEmotionsDataset['train'].features['labels'].feature.names

# 2. bucket mapping
sentimentMap = {
  "Positive": ["admiration", "amusement", "approval", "caring", "desire", "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"],
  "Negative": ["anger", "annoyance", "disappointment", "disapproval", "disgust", "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"],
  "Neutral": ["neutral", "realization", "surprise", "curiosity", "confusion"]
}

def get_true_bucket(label_indices):
    pos_score = sum(1 for idx in label_indices if sentimentLabels[idx] in sentimentMap['Positive'])
    neg_score = sum(1 for idx in label_indices if sentimentLabels[idx] in sentimentMap['Negative'])
    neu_score = sum(1 for idx in label_indices if sentimentLabels[idx] in sentimentMap['Neutral'])
    
    scores = {'Positive': pos_score, 'Negative': neg_score, 'Neutral': neu_score}
    return max(scores, key=scores.get)

print("Dataset loaded and bucket mapping ready.")

Dataset loaded and bucket mapping ready.


In [3]:
def setup_model():
    genai.configure(api_key=API_KEY)
    
    generation_config = {
        "temperature": 0.0, 
        "top_p": 0.95,
        "max_output_tokens": 1024,
        "response_mime_type": "application/json",
    }
    
    # Disable safety blocks to prevent dropped political comments
    safety_settings = {
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }

    # FEW-SHOT PROMPT
    system_instruction = (
        "You are an expert sentiment classification system. Your task is to analyze Reddit comments "
        "and classify their emotional sentiment into exactly one of three buckets: 'Positive', 'Negative', or 'Neutral'.\n\n"
        "Here are some examples of how to classify:\n"
        "Comment: \"I absolutely love the new design! Great job to the devs.\"\n"
        "Sentiment: Positive\n\n"
        "Comment: \"This is the worst update ever, completely broken and unplayable.\"\n"
        "Sentiment: Negative\n\n"
        "Comment: \"The patch notes were released yesterday afternoon.\"\n"
        "Sentiment: Neutral\n\n"
        "Comment: \"I'm so frustrated with how long this is taking to resolve.\"\n"
        "Sentiment: Negative\n\n"
        "Comment: \"Wow, I am so incredibly happy for you! Congratulations!\"\n"
        "Sentiment: Positive\n\n"
        "Return a JSON object with a single key 'sentiment' containing either 'Positive', 'Negative', or 'Neutral'."
    )
    
    return genai.GenerativeModel(
        model_name='gemini-2.0-flash',
        generation_config=generation_config,
        safety_settings=safety_settings,
        system_instruction=system_instruction
    )

def analyze_comment(model, body):
    prompt = f"RequestID: {str(uuid.uuid4())}\nComment Body: {body}"
    
    for attempt in range(3):
        try:
            response = model.generate_content(prompt)
            if not response.parts:
                return "Neutral"
            
            text_content = response.text.strip()
            if "```" in text_content:
                text_content = re.sub(r'```json\s*|\s*```', '', text_content)
            
            result = json.loads(text_content)
            sentiment = result.get("sentiment", "Neutral").capitalize()
            
            return sentiment if sentiment in ["Positive", "Negative", "Neutral"] else "Neutral"

        except Exception as e:
            time.sleep(2) 
            
    return "Neutral"

model = setup_model()
print("Gemini initialized with few-shot prompt.")

Gemini initialized with few-shot prompt.


In [ ]:
llm_predictions = []
true_buckets = []

NUM_SAMPLES = len(goEmotionsDataset['test']) 

print(f"Running Few-Shot Baseline on {NUM_SAMPLES} GoEmotions test samples...")
start_time = time.time()

for i in tqdm(range(NUM_SAMPLES)):
    # 1. Pull text and true labels
    comment_text = goEmotionsDataset['test'][i]['text']
    true_raw_labels = goEmotionsDataset['test'][i]['labels'] 
    
    # 2. Get predictions and ground truth
    prediction = analyze_comment(model, body=comment_text)
    ground_truth = get_true_bucket(true_raw_labels)
    
    llm_predictions.append(prediction)
    true_buckets.append(ground_truth)
    
    # Rate limit protection (Adjust if you have a higher tier API key)
    time.sleep(1.0) 

end_time = time.time()
print(f"Inference complete! Total time: {end_time - start_time:.2f} seconds.")

Running Few-Shot Baseline on 200 GoEmotions test samples...


100%|██████████| 200/200 [05:10<00:00,  1.55s/it]

Inference complete! Total time: 310.88 seconds.


In [5]:
print("\nFew-Shot LLM Baseline Evaluation (3 Buckets):")
print("-" * 50)
print(classification_report(
    true_buckets,
    llm_predictions,
    labels=['Positive', 'Negative', 'Neutral'] 
))


Few-Shot LLM Baseline Evaluation (3 Buckets):
--------------------------------------------------
              precision    recall  f1-score   support

    Positive       0.80      0.49      0.61        71
    Negative       0.50      0.59      0.54        51
     Neutral       0.50      0.62      0.55        78

    accuracy                           0.56       200
   macro avg       0.60      0.57      0.57       200
weighted avg       0.60      0.56      0.57       200

